# Build the dense FAISS index (Colab GPU)

In [ ]:
import os
from getpass import getpass
os.environ["GH_TOKEN"] = getpass("Paste GitHub token: ")
os.environ["HF_TOKEN"] = getpass("Paste Hugging Face token: ")

In [ ]:
# Colab GPU + repo setup (run once)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable GPU: Runtime → Change runtime type → T4 GPU")
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://${GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
%pip install -e .

In [ ]:
# Re-run after pushing code/config changes
!git pull
%pip install .

In [ ]:
# Fetch the canonical processed corpus (same gdrive the training notebooks use)
!gdown 1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
!mkdir -p data/processed
!unzip -o data_processed.zip -d data/processed
!rm -rf data/processed/__MACOSX data_processed.zip
import pandas as pd
print("corpus rows:", len(pd.read_csv("data/processed/corpus.csv")), "(expect 262168)")

In [ ]:
!pip install -U huggingface_hub
!hf auth login --token ${HF_TOKEN}

## Build the index

In [ ]:
import yaml
import pandas as pd
from app.config import load_config
from vnlegal_rag_v2.factory import build_pipeline
from vnlegal_rag_v2.utils.device import get_device
from app.index_cache import load_or_build_index

print("device:", get_device())
cfg = load_config()                       # configs/app/app.yaml (full corpus, dense-only)
print("corpus:", cfg["corpus_path"], "| index_dir:", cfg["index_dir"])

df = pd.read_csv(cfg["corpus_path"])
documents = df["text"].astype(str).tolist()
cids = df["cid"].tolist()
print(f"corpus: {len(documents):,} docs")

pipeline = build_pipeline(yaml.safe_load(open(cfg["pipeline_config"])))
print("retriever weights:", pipeline.retriever.weights, "(dense-only → bm25 skipped)")

hit = load_or_build_index(pipeline, documents, cids, cfg["corpus_path"], cfg["index_dir"])
print("\n[build] " + ("cache hit — already built" if hit else "built + cached to .app_index/"))

## Verify

In [ ]:
import os
p2 = build_pipeline(yaml.safe_load(open(cfg["pipeline_config"])))
hit2 = load_or_build_index(p2, documents, cids, cfg["corpus_path"], cfg["index_dir"])
print("reload cache hit:", hit2, "(expect True)")
print("retrieve top-5 (‘thời hiệu khởi kiện vụ án dân sự’):",
      p2.retrieve(["thời hiệu khởi kiện vụ án dân sự"], top_k=5)[0])
print("\nindex artifacts:")
for root, _, files in os.walk(cfg["index_dir"]):
    for fn in files:
        p = os.path.join(root, fn)
        print(f"  {os.path.getsize(p)/1e6:8.1f} MB  {p}")

## Download to local

In [ ]:
!zip -r index.zip .app_index


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mv index.zip /content/drive/MyDrive/index.zip